# Fine-tune IndicBART for Hindi Summarization (Kaggle)

This notebook fine-tunes [`ai4bharat/IndicBARTSS`](https://huggingface.co/ai4bharat/IndicBARTSS) on a sample of [XL-Sum](https://huggingface.co/datasets/csebuetnlp/xlsum) for **Hindi**. XL-Sum provides news article → summary pairs in 44 languages.

### How to run on Kaggle
1. Create a new Notebook at [kaggle.com](https://www.kaggle.com).
2. **Settings → Accelerator → GPU T4 x2** (any GPU works).
3. **Settings → Internet → On** (needed to download models/data).
4. Run **Run All** to train the Hindi model.

The weights are saved under `./indicbart-finetuned-hindi`. Place them in `models/hindi/` in the repo.

In [ ]:
# Install training dependencies.
!pip install -q transformers datasets evaluate rouge_score nltk accelerate sentencepiece


In [ ]:
import nltk
import numpy as np
import torch

import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

nltk.download("punkt")

print("CUDA available:", torch.cuda.is_available())

# ===== EDIT ME =====
LANG = "hindi"
LANG_CODE = "hi_IN"
# ===================

MODEL_ID = "ai4bharat/IndicBARTSS"
DATASET = "csebuetnlp/xlsum"

# Small subset for a fast demo run; remove .select() to use the full data.
TRAIN_SIZE = 2000
EVAL_SIZE = 200

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128


In [ ]:
# XL-Sum is script-based (unsupported in datasets>=4), so load its Parquet shards directly.
BASE = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/refs%2Fconvert%2Fparquet"
if LANG == "hindi":
    _files = {
        "train": [f"{BASE}/hindi/train/0000.parquet", f"{BASE}/hindi/train/0001.parquet"],
        "validation": [f"{BASE}/hindi/validation/0000.parquet"],
        "test": [f"{BASE}/hindi/test/0000.parquet"],
    }
dataset = load_dataset("parquet", data_files=_files)

train = dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
valid = dataset["validation"].shuffle(seed=42).select(range(EVAL_SIZE))
test = dataset["test"].shuffle(seed=42).select(range(EVAL_SIZE))

print("Article :", train[0]["text"][:200])
print("---")
print("Summary :", train[0]["summary"][:200])


In [ ]:
# IndicBARTSS uses a SentencePiece tokenizer; fast tokenizer is unavailable.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, do_lower_case=False, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)


In [ ]:
def get_decoder_start_id(tokenizer, lang_code):
    """Resolve the forced decoder-start token id (mBART-style or <2xx>-style)."""
    lang_to_id = getattr(tokenizer, "lang_code_to_id", None) or {}
    if lang_code in lang_to_id:
        return int(lang_to_id[lang_code])
    token_id = tokenizer.convert_tokens_to_ids(f"<2{lang_code[:2]}>")
    if token_id != tokenizer.unk_token_id:
        return int(token_id)
    return tokenizer.bos_token_id


# Pin the decoder language for BOTH training (shift) and generation.
_start_id = get_decoder_start_id(tokenizer, LANG_CODE)
model.config.decoder_start_token_id = _start_id
model.generation_config.forced_bos_token_id = _start_id
print("decoder_start_token_id =", _start_id)


In [ ]:
def preprocess_function(examples):
    lang_tok = f"<2{LANG_CODE[:2]}>"
    inputs = [f"{t} </s> {lang_tok}" for t in examples["text"]]
    targets = [f"{lang_tok} {s} </s>" for s in examples["summary"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True, add_special_tokens=False)
    labels = tokenizer(targets, max_length=MAX_TARGET_LENGTH, truncation=True, add_special_tokens=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train.map(preprocess_function, batched=True, remove_columns=train.column_names)
tokenized_valid = valid.map(preprocess_function, batched=True, remove_columns=valid.column_names)


In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

rouge = evaluate.load("rouge")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    pad = int(tokenizer.pad_token_id)

    # Drop any negative/invalid sentinel id (e.g. -100) before decoding, so the
    # tokenizer never sees an out-of-range value on any transformers version.
    def clean_ids(seqs):
        if hasattr(seqs, "tolist"):
            seqs = seqs.tolist()
        out = []
        for seq in seqs:
            cleaned = []
            for t in seq:
                try:
                    v = int(t)
                except Exception:
                    continue
                if v < 0:
                    continue
                cleaned.append(v)
            out.append(cleaned)
        return out

    preds = clean_ids(predictions)
    labs = clean_ids(labels)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labs, skip_special_tokens=True)

    decoded_preds = ["\n".join(nltk.sent_tokenize(p.strip())) for p in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(l.strip())) for l in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 2) for k, v in result.items()}
    result["gen_len"] = round(float(np.mean([sum(1 for t in p if t != pad) for p in preds])), 2)
    return result


In [ ]:
output_dir = f"./indicbart-finetuned-{LANG}"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()


In [ ]:
metrics = trainer.evaluate(eval_dataset=tokenized_valid, max_length=MAX_TARGET_LENGTH, num_beams=4)
print(metrics)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Saved to", output_dir)


In [ ]:
# Try it on a held-out example.
article = test[0]["text"]
inputs = tokenizer(article, max_length=MAX_INPUT_LENGTH, truncation=True, return_tensors="pt").to(model.device)
summary_ids = model.generate(**inputs, max_length=128, min_length=20, num_beams=4, early_stopping=True)
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


### Use it in the app
1. Download `indicbart-finetuned-hindi` from Kaggle → **Output** tab.
2. Place it at `models/hindi/` in the repo.
3. The app auto-detects it and uses it instead of the public checkpoint.